##Task2:


In [1]:
from dataclasses import dataclass
from collections import deque


@dataclass(frozen=True)
class Action:
    name: str
    positive_preconditions: frozenset
    negative_preconditions: frozenset
    positive_effects: frozenset
    negative_effects: frozenset


def is_applicable(action, state):
    return (
        action.positive_preconditions <= state
        and action.negative_preconditions.isdisjoint(state)
    )


def apply_action(action, state):
    new_state = set(state)

    # Remove negative effects
    new_state.difference_update(action.negative_effects)

    # Add positive effects
    new_state.update(action.positive_effects)

    return frozenset(new_state)


def bfs_planner(initial_state, actions, goal):
    frontier = deque([(initial_state, [])])
    visited = {initial_state}

    while frontier:
        current_state, plan = frontier.popleft()

        # Goal test
        if goal <= current_state:
            return plan, current_state

        for action in actions:

            if is_applicable(action, current_state):

                next_state = apply_action(
                    action,
                    current_state
                )

                if next_state not in visited:
                    visited.add(next_state)

                    frontier.append(
                        (next_state, plan + [action])
                    )

    return None, None

In [2]:
actions = []

# Move actions
move_connections = [
    ("A", "B"),
    ("B", "A"),
    ("B", "C"),
    ("C", "B")
]

for source, destination in move_connections:
    actions.append(
        Action(
            name=f"Move({source},{destination})",
            positive_preconditions=frozenset({
                f"At(Robot,{source})"
            }),
            negative_preconditions=frozenset(),
            positive_effects=frozenset({
                f"At(Robot,{destination})"
            }),
            negative_effects=frozenset({
                f"At(Robot,{source})"
            })
        )
    )

In [3]:
for location in ["A", "B", "C"]:
    actions.append(
        Action(
            name=f"PickUp(Package,{location})",
            positive_preconditions=frozenset({
                f"At(Robot,{location})",
                f"At(Package,{location})"
            }),
            negative_preconditions=frozenset({
                "Holding(Package)"
            }),
            positive_effects=frozenset({
                "Holding(Package)"
            }),
            negative_effects=frozenset({
                f"At(Package,{location})"
            })
        )
    )

In [4]:
for location in ["A", "B", "C"]:
    actions.append(
        Action(
            name=f"Drop(Package,{location})",
            positive_preconditions=frozenset({
                f"At(Robot,{location})",
                "Holding(Package)"
            }),
            negative_preconditions=frozenset(),
            positive_effects=frozenset({
                f"At(Package,{location})"
            }),
            negative_effects=frozenset({
                "Holding(Package)"
            })
        )
    )

In [6]:
initial_state = frozenset({
    "At(Robot,A)",
    "At(Package,A)"
})

goal = frozenset({
    "At(Package,C)"
})

plan, final_state = bfs_planner(
    initial_state,
    actions,
    goal
)

if plan is None:
    print("No plan found.")
else:
    print("Plan found!")
    print()

    print("Initial state:")
    print(sorted(initial_state))
    print()

    current_state = initial_state

    for i, action in enumerate(plan, start=1):

        current_state = apply_action(
            action,
            current_state
        )

        print(f"Action {i}: {action.name}")
        print("State:", sorted(current_state))
        print()

    print("Final state:")
    print(sorted(final_state))

Plan found!

Initial state:
['At(Package,A)', 'At(Robot,A)']

Action 1: PickUp(Package,A)
State: ['At(Robot,A)', 'Holding(Package)']

Action 2: Move(A,B)
State: ['At(Robot,B)', 'Holding(Package)']

Action 3: Move(B,C)
State: ['At(Robot,C)', 'Holding(Package)']

Action 4: Drop(Package,C)
State: ['At(Package,C)', 'At(Robot,C)']

Final state:
['At(Package,C)', 'At(Robot,C)']


##Task3:

In [7]:
# Test A: Solvable Problem

initial_state_A = frozenset({
    "At(Robot,A)",
    "At(Package,A)"
})

goal_A = frozenset({
    "At(Package,C)"
})

plan_A, final_state_A = bfs_planner(
    initial_state_A,
    actions,
    goal_A
)

print("Test A: Solvable Problem")
print("Initial state:", sorted(initial_state_A))
print("Goal:", sorted(goal_A))
print()

if plan_A is None:
    print("Plan found: No")
else:
    print("Plan found: Yes")
    print("Plan:")

    current_state = initial_state_A

    for i, action in enumerate(plan_A, start=1):
        current_state = apply_action(action, current_state)
        print(f"{i}. {action.name}")

    print()
    print("Final state:", sorted(current_state))

Test A: Solvable Problem
Initial state: ['At(Package,A)', 'At(Robot,A)']
Goal: ['At(Package,C)']

Plan found: Yes
Plan:
1. PickUp(Package,A)
2. Move(A,B)
3. Move(B,C)
4. Drop(Package,C)

Final state: ['At(Package,C)', 'At(Robot,C)']


In [8]:
def verify_plan(initial_state, plan, goal):
    current_state = initial_state

    for action in plan:
        if not is_applicable(action, current_state):
            return False, current_state, action

        current_state = apply_action(action, current_state)

    return goal <= current_state, current_state, None


valid_A, final_A, invalid_action_A = verify_plan(
    initial_state_A,
    plan_A,
    goal_A
)

print("Plan valid:", valid_A)

if invalid_action_A is not None:
    print("Invalid action:", invalid_action_A.name)

Plan valid: True


In [9]:
# Test B: Impossible Problem
# Remove all PickUp actions

actions_without_pickup = [
    action
    for action in actions
    if not action.name.startswith("PickUp")
]

initial_state_B = frozenset({
    "At(Robot,A)",
    "At(Package,A)"
})

goal_B = frozenset({
    "At(Package,C)"
})

plan_B, final_state_B = bfs_planner(
    initial_state_B,
    actions_without_pickup,
    goal_B
)

print("Test B: Impossible Problem")
print("Initial state:", sorted(initial_state_B))
print("Goal:", sorted(goal_B))
print()

if plan_B is None:
    print("Plan found: No")
    print("No plan found.")
else:
    print("Plan found: Yes")
    print("Plan:")

    for i, action in enumerate(plan_B, start=1):
        print(f"{i}. {action.name}")


Test B: Impossible Problem
Initial state: ['At(Package,A)', 'At(Robot,A)']
Goal: ['At(Package,C)']

Plan found: No
No plan found.


In [10]:
# Test C: Irrelevant Actions

irrelevant_actions = [
    Action(
        name="MoveRobot(A,B)",
        positive_preconditions=frozenset({
            "At(Robot,A)"
        }),
        negative_preconditions=frozenset(),
        positive_effects=frozenset({
            "At(Robot,B)"
        }),
        negative_effects=frozenset({
            "At(Robot,A)"
        })
    ),

    Action(
        name="MoveRobot(B,C)",
        positive_preconditions=frozenset({
            "At(Robot,B)"
        }),
        negative_preconditions=frozenset(),
        positive_effects=frozenset({
            "At(Robot,C)"
        }),
        negative_effects=frozenset({
            "At(Robot,B)"
        })
    )
]

actions_with_irrelevant = actions + irrelevant_actions

initial_state_C = frozenset({
    "At(Robot,A)",
    "At(Package,A)"
})

goal_C = frozenset({
    "At(Package,C)"
})

plan_C, final_state_C = bfs_planner(
    initial_state_C,
    actions_with_irrelevant,
    goal_C
)

print("Test C: Irrelevant Actions")
print("Initial state:", sorted(initial_state_C))
print("Goal:", sorted(goal_C))
print()

if plan_C is None:
    print("Plan found: No")
else:
    print("Plan found: Yes")
    print("Plan:")

    current_state = initial_state_C

    for i, action in enumerate(plan_C, start=1):
        current_state = apply_action(action, current_state)
        print(f"{i}. {action.name}")
        print("   State:", sorted(current_state))

    print()
    print("Final state:", sorted(current_state))

Test C: Irrelevant Actions
Initial state: ['At(Package,A)', 'At(Robot,A)']
Goal: ['At(Package,C)']

Plan found: Yes
Plan:
1. PickUp(Package,A)
   State: ['At(Robot,A)', 'Holding(Package)']
2. Move(A,B)
   State: ['At(Robot,B)', 'Holding(Package)']
3. Move(B,C)
   State: ['At(Robot,C)', 'Holding(Package)']
4. Drop(Package,C)
   State: ['At(Package,C)', 'At(Robot,C)']

Final state: ['At(Package,C)', 'At(Robot,C)']
